In [36]:
import requests
import hashlib
import pandas as pd

In [37]:
def load_json_from_github(path_from_root: str, branch: str="main"):
    """
    Retrieves JSON files from the Accenture 1O repository
    :param path_from_root: the path to the file from the branch's root
    :param branch: the branch the file is located on, assumes "main" branch
    :return: the file as a json object
    """
    url = f"https://raw.githubusercontent.com/Break-Through-Tech/Accenture-1O-contract-review-challenge/{branch}/{path_from_root}"

    response = requests.get(url)
    response.raise_for_status()
    return response.json()

data = load_json_from_github(
    path_from_root="data/cuad/train_separate_questions.json",
    branch="setup-and-explore"
)
raw_data = data["data"]



# Normalizing Data

This is the first step for data preprocessing. In order to run NLP and ML techniques on the CUAD dataset, it must be normalized into a standardized structure. This structure is as follows:
```python
{
    "contracts": pd.DataFrame,
    "documents": pd.DataFrame,
    "categories": pd.DataFrame,
    "annotation_sets": pd.DataFrame,
    "spans": pd.DataFrame,
}
```
Contracts include...
- `contract_id`: an enumerated key
- `title`: the title of the contract

Documents include...
- `document_id`: an enumerated key
- `contract_id`: foreign key linked to `contracts.contract_id`
- `context`: the complete, unchanged contract text
- `context_group_id`: a hash code for the context

Categories include...
- `category_id`: the category name in snake case
- `category_name`: a human-readable formatted `category_id`
- `question`: the question asked to identify key clauses

Annotation Sets include...
- `annotation_set_id`: a key consisting of the `contract_id` and `category_id`
- `contract_id`: foreign key linked to `contracts.contract_id`
- `category_id`: foreign key linked to `categories.category_id`
- `is_impossible`: boolean indicating whether an answer exists to the asked question

Spans include...
- `span_id`: a key consisting of the `contract_id` and `category_id` and span number
- `annotation_set_id`: a foreign key linked to `annotation_sets.annotation_set_id`
- `source_qa_id`: the contract's original `title` and `category_id`
- `answer_text`: the identified clause in plain text
- `answer_start`: the starting index of the `answer_text`
- `answer_end`: the ending index of the `answer_text`
-

In [38]:
def normalize_cuad(data: dict) -> dict:
    """
    Normalizes the given CUAD dataset from nested JSON objects into separate actionable DataFrames
    :param data: list of document objects from the CUAD dataset
    :return: a single dictionary that contains separate DataFrames for contracts, documents, categories, annotation sets, and spans. These DataFrames are connected via 'foreign keys' (aka their IDs)
    """

    # assign contract ids for each contract in the dataset
    for i, item in enumerate(data):
        item["contract_id"] = f"contract_{i + 1:04d}"

    # get each document and their respective data (title, context, qas)
    documents = pd.json_normalize(
        data,
        record_path="paragraphs",
        meta=["title", "contract_id"]
    )[["contract_id", "title", "context"]]

    # assign ids and hashes
    documents["document_id"] = [f"document_{i + 1:04d}" for i in range(len(documents))]
    documents["context_group_id"] = documents["context"].apply(
        lambda c: "context_" + hashlib.md5(c.encode("utf-8")).hexdigest()[:10]
    )

    # create the contracts "table" and remove duplicates
    contracts = (
        documents[["contract_id", "title"]]
        .drop_duplicates(subset="contract_id")
        .reset_index(drop=True)
    )

    # ensures all indexes are correct
    documents = documents[
        ["document_id", "contract_id", "context", "context_group_id"]
    ].reset_index(drop=True)

    # get all Q&A objects
    qas = pd.json_normalize(
        data,
        record_path=["paragraphs", "qas"],
        meta=["title", "contract_id"]
    )

    qas["category_name"] = qas["question"].str.extract(f'"([^"]+)')
    qas["category_id"] = (
        qas["category_name"]
        .str.lower()
        .str.replace(r"[^a-z0-9]+", "_", regex=True)
        .str.strip("_")
    )
    qas["annotation_set_id"] = qas["contract_id"] + "__" + qas['category_id'].str.lower()

    # get categories. each question typically corresponds to a category
    categories = (
        qas[["category_id", "category_name", "question"]]
        .drop_duplicates(subset="category_id")
        .reset_index(drop=True)
    )

    # get the annotation set: one contract paired with one clause category/question
    annotation_sets = (
        qas.groupby(
            ["annotation_set_id", "contract_id", "category_id"]
        , as_index=False)["is_impossible"]
        .all()
    )

    # get each span: one individual answer inside the Q&A's answer list
    spans = qas[["answers", "id", "annotation_set_id"]].explode("answers")
    spans = spans[spans["answers"].notna()].copy() # drops impossible rows

    # expand the answer fields to be included in the top-level columns
    answer_fields = pd.json_normalize(spans["answers"])
    spans = pd.concat([spans.drop(columns="answers"), answer_fields], axis=1)

    spans = spans.rename(columns={"id": "source_qa_id", "text": "answer_text"})
    spans["answer_end"] = spans["answer_start"] + spans["answer_text"].str.len()

    # create the span id
    num_spans = spans.groupby("annotation_set_id").cumcount()
    spans["span_id"] = (
        spans["annotation_set_id"] + "__span_" + num_spans.astype(str).str.zfill(3)
    )
    spans = spans[[
        "span_id",
        "annotation_set_id",
        "source_qa_id",
        "answer_text",
        "answer_start",
        "answer_end",
    ]].reset_index(drop=True)

    return {
        "contracts": contracts,
        "documents": documents,
        "categories": categories,
        "annotation_sets": annotation_sets,
        "spans": spans,
    }



# Verifying Counts

In [39]:
normalized_data = normalize_cuad(raw_data)


In [40]:
totalContracts = 408
totalDocuments = 408
totalCategories = 41
totalAnnotationSets = 16728
totalSpans = 11180

contracts = normalized_data["contracts"]
documents = normalized_data["documents"]
categories = normalized_data["categories"]
annotation_sets = normalized_data["annotation_sets"]
spans = normalized_data["spans"]

In [41]:
print(f"{totalContracts} Contracts \t\t\t\t{"PASSED" if contracts.shape[0] == totalContracts else "FAILED"}")
print(f"{totalDocuments} Documents \t\t\t\t{"PASSED" if documents.shape[0] == totalDocuments else "FAILED"}")
print(f"{totalCategories} Categories \t\t\t\t{"PASSED" if categories.shape[0] == totalCategories else "FAILED"}")
print(f"{totalAnnotationSets} AnnotationSets \t\t{"PASSED" if annotation_sets.shape[0] == totalAnnotationSets else "FAILED"}")
print(f"{totalSpans} Spans \t\t\t\t{"PASSED" if spans.shape[0] == totalSpans else "FAILED"}")


408 Contracts 				PASSED
408 Documents 				PASSED
41 Categories 				PASSED
16728 AnnotationSets 		PASSED
11180 Spans 				PASSED


# Exploratory Data Analysis

## Contract/document length statistics


In [42]:
documents.head()

,document_id,contract_id,context,context_group_id
0,document_0001,contract_0001,EXHIBIT 10.6\n\n ...,context_41f7921a65
1,document_0002,contract_0002,Exhibit 10.26 CONFIDENTIAL TREATMENT HAS BE...,context_d545018697
2,document_0003,contract_0003,Exhibit 1\n\nJOINT FILING AGREEMENT\n\nThe und...,context_8cc65b1516
3,document_0004,contract_0004,REDACTED COPY\n\nCONFIDENTIAL TREATMENT REQUES...,context_3b2346a2bc
4,document_0005,contract_0005,Exhibit 10.23 Corporate Address Fannin South P...,context_1d348d1f41


In [43]:
documents.columns

Index(['document_id', 'contract_id', 'context', 'context_group_id'], dtype='str')

In [44]:
documents['document_length'] = documents['context'].str.len()

In [45]:
documents['document_length'].describe()

count       408.000000
mean      53991.710784
std       55985.180326
min        1081.000000
25%       16569.750000
50%       35960.500000
75%       68439.750000
max      338211.000000
Name: document_length, dtype: float64

## Document Length Statistics

In [46]:
annotation_sets.columns

Index(['annotation_set_id', 'contract_id', 'category_id', 'is_impossible'], dtype='str')

In [47]:
annotation_sets['category_id'].value_counts()

category_id
affiliate_license_licensee            408
affiliate_license_licensor            408
agreement_date                        408
anti_assignment                       408
audit_rights                          408
cap_on_liability                      408
change_of_control                     408
competitive_restriction_exception     408
covenant_not_to_sue                   408
document_name                         408
effective_date                        408
exclusivity                           408
expiration_date                       408
governing_law                         408
insurance                             408
ip_ownership_assignment               408
irrevocable_or_perpetual_license      408
joint_ip_ownership                    408
license_grant                         408
liquidated_damages                    408
minimum_commitment                    408
most_favored_nation                   408
no_solicit_of_customers               408
no_solicit_of_employee

In [48]:
annotation_sets['is_impossible'].value_counts()

is_impossible
True     11270
False     5458
Name: count, dtype: int64